<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 09 · E9 — Distractor discrimination

With 7 irrelevant tables present (3 adversarial — colliding column names / shared
FKs), does onboarding/enrichment/query **ignore** the tables the glossary never
names? We measure table-selection precision/recall and query-time distractor touch.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import eval_v2 as v2
import seagate_scoring as scoring

FIXTURE = 'seagate_multi'
fdir = v2.fixture_dir(FIXTURE)
manifest = v2.load_table_manifest(FIXTURE)

# Multi-schema project: primary schema seagate_ops + the seagate_core scope.
# seagate_ref is deliberately EXCLUDED (pulling it in is an E9 failure).
config = ec.EvalConfig.from_env(
    schema_name='seagate_ops',
    results_dir=fdir.parent.parent / 'evaluation' / 'results' / 'seagate_multi',
)
client = v2.AgentClientV2(config, schema_names=['seagate_core', 'seagate_ops'])
me = client.login()
print('Authenticated as:', me.get('username') or me)

# Preconditions: Postgres required (R4); memory loop must be off (R1, warn-only).
pre = client.assert_eval_preconditions(require_postgres=True)
print('DB backend:', pre['backend'])
for w in pre['warnings']:
    print('WARNING:', w)

questions = ec.parse_test_queries(fdir / 'test_queries.md')
print('Loaded', len(questions), 'graded questions (Q1-Q18)')
glossary = (fdir / 'bi_glossary.md').read_text(encoding='utf-8')

Authenticated as: admin


DB backend: postgresql
Loaded 18 graded questions (Q1-Q18)


In [2]:
def score_sweep(results):
    """Score a result set with the exact ground-truth scorer (incl. Q16-Q18)."""
    verdicts = {r['id']: scoring.score_result(
        r['id'], r['result_rows'], r['answer_summary']) for r in results}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return correct, verdicts

### Onboard + enrich, then measure table selection vs the R/D sets

In [3]:
print('archived', client.clean_baseline(), 'project(s)')
project = client.resolve_project(); pid = project['id']
client.onboard(pid)
print('after onboard:', client.selection_metrics(pid, manifest))

doc = client.create_document_from_text(pid, glossary, 'bi_glossary.md')
client.enrich_round(pid, doc['id'], wait_coverage=False)
metrics = client.selection_metrics(pid, manifest)
print('after enrich :', metrics)

# Scope check: the out-of-scope seagate_ref schema must NOT be pulled in.
project = client.resolve_project()
pulled = client.out_of_scope_schema_pulled_in(project, manifest)
print('out-of-scope schemas erroneously included:', pulled or 'none (good)')

archived 1 project(s)


after onboard: {'precision': 0.583, 'recall': 1.0, 'f1': 0.737, 'selected_relevant': ['seagate_drive_skus', 'seagate_production_events', 'seagate_production_lines', 'seagate_quality_tests', 'seagate_shipments', 'seagate_sites', 'seagate_work_orders'], 'distractor_inclusions': ['seagate_finance_ledger', 'seagate_hr_roster', 'seagate_iot_sensor_logs', 'seagate_maintenance_logs', 'seagate_vendor_contracts'], 'distractor_inclusion_rate': 0.714, 'missed_relevant': []}


AgentError: PATCH mdl-files -> 422: {"detail":"Cannot activate an MDL file with validation errors: Duplicate model name: seagate_production_events.; Duplicate model name: seagate_production_events."}

### Query-time leakage: does any emitted SQL touch a distractor table?

In [4]:
results = ec.run_experiment(client, 'wren_bi', questions, save=False)
distractors = manifest['distractor_tables']
leaks = []
for r in results:
    touched = v2.sql_references_tables(r['sql'], distractors)
    if touched:
        leaks.append((r['id'], touched))
print('distractor touches:', leaks or 'none')
print(f'distractor-touch rate: {len(leaks)}/{len(results)}')
correct, _ = score_sweep(results)
print(f'graded correct (sanity): {correct}/18')

distractor touches: [('Q5', ['seagate_iot_sensor_logs']), ('Q13', ['seagate_finance_ledger']), ('Q18', ['seagate_finance_ledger'])]
distractor-touch rate: 3/18
graded correct (sanity): 5/18


**Adversarial focus.** Inspect whether `seagate_finance_ledger` (decoy `units`),
`seagate_iot_sensor_logs` (decoy `temperature_c` + shared `line_id`), or
`seagate_hr_roster` (decoy `shift_code` + shared `site_id`) specifically got
mis-selected — those are the hard cases. A non-empty `distractor_inclusions` or any
SQL leak is the finding to report, per-table (not just the aggregate).